# 01 · BKT starter: ASSISTments 2009 → first Bayesian Knowledge Tracing model

Part of: *Offline-First Personalised Learning Path for WAEC/JAMB Mathematics Using Knowledge Tracing*

What this notebook does:

1. Loads and cleans **ASSISTments 2009–2010 Skill Builder** data (or a synthetic stand-in with the same columns).
2. Splits **by student** (so test students are unseen: the same situation as your Nigerian students).
3. Trains baselines (global mean, per-skill mean).
4. Trains BKT two ways:
   - **pyBKT** (the library in your plan), and
   - a **from-scratch NumPy BKT**: about 40 lines, the same equations you will run in JavaScript inside the PWA.
5. Evaluates AUC / RMSE / accuracy / log-loss on held-out students.
6. Exports per-skill parameters to JSON for on-device use, and demos a simple "next question" policy against a fixed order.

**Getting the data:** search for "ASSISTments 2009-2010 skill builder data". Download `skill_builder_data_corrected_collapsed.csv` (preferred) or `skill_builder_data.csv`, put it in `data/`, and set `USE_SYNTHETIC = False` below.

## 0 · Setup

In [1]:
# pip install pandas numpy scikit-learn scipy matplotlib pyBKT mlflow
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, mean_squared_error, accuracy_score, log_loss

warnings.filterwarnings("ignore", category=RuntimeWarning)
SEED = 42
rng = np.random.default_rng(SEED)

# ---- configuration -------------------------------------------------------
USE_SYNTHETIC = False                        # set False once the real CSV is in data/
DATA_PATH = Path("data/skill_builder_data_corrected_collapsed.csv")
OUT_DIR = Path("artifacts"); OUT_DIR.mkdir(exist_ok=True)
MIN_INTERACTIONS_PER_SKILL = 50             # drop very rare skills
TEST_FRACTION = 0.2

try:
    from pyBKT.models import Model as PyBKTModel
    HAVE_PYBKT = True
except ImportError:
    HAVE_PYBKT = False
try:
    import mlflow
    HAVE_MLFLOW = True
except ImportError:
    HAVE_MLFLOW = False
print(f"pyBKT available: {HAVE_PYBKT} | MLflow available: {HAVE_MLFLOW}")

pyBKT available: True | MLflow available: True


## 1 · Data

### 1a · Synthetic stand-in (for testing the pipeline before you have the CSV)
The data is generated *from* a BKT model with known parameters, so §5 can check that fitting **recovers** them. That is a useful sanity check to report in your write-up.

In [2]:
def make_synthetic(n_students=800, n_skills=12, seed=SEED):
    r = np.random.default_rng(seed)
    true = {}
    for k in range(n_skills):
        true[f"skill_{k:02d}"] = dict(
            prior=r.uniform(0.1, 0.6), learn=r.uniform(0.05, 0.35),
            guess=r.uniform(0.1, 0.3), slip=r.uniform(0.05, 0.2), forget=0.0)
    rows, order_id = [], 0
    for u in range(n_students):
        skills = r.choice(list(true), size=r.integers(2, 7), replace=False)
        for s in skills:
            p = true[s]
            known = r.random() < p["prior"]
            for _ in range(r.integers(3, 25)):
                p_correct = (1 - p["slip"]) if known else p["guess"]
                rows.append((order_id, u, s, int(r.random() < p_correct), 1))
                order_id += 1
                if not known and r.random() < p["learn"]:
                    known = True
    df = pd.DataFrame(rows, columns=["order_id", "user_id", "skill_name", "correct", "original"])
    return df, true

if USE_SYNTHETIC:
    raw, TRUE_PARAMS = make_synthetic()
else:
    TRUE_PARAMS = None
    raw = pd.read_csv(DATA_PATH, encoding="latin-1", low_memory=False)
print(raw.shape); raw.head()

(525534, 30)


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.00000
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.00000
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.00000
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.00000
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.00000,3,3.00000


### 1b · Cleaning
Standard choices for ASSISTments 2009 (write them down in your methods section, because published AUCs vary a lot with preprocessing):

- drop rows with no skill;
- keep **original** problems only (`original == 1`), excluding scaffolding sub-questions;
- keep `correct` in {0, 1} (first attempt correct or not);
- remove exact duplicates of `(order_id, skill_name)` (a known issue in the original file). A multi-skill problem still counts once **per skill**, which is the usual BKT treatment;
- drop skills with fewer than `MIN_INTERACTIONS_PER_SKILL` rows;
- sort by `order_id`, which is the chronological order.

In [3]:
def clean(df):
    n0 = len(df)
    df = df.dropna(subset=["skill_name"]).copy()
    if "original" in df.columns:
        df = df[df["original"] == 1]
    df = df[df["correct"].isin([0, 1])]
    df["correct"] = df["correct"].astype(int)
    df["skill_name"] = df["skill_name"].astype(str).str.strip()
    df = df.drop_duplicates(subset=["order_id", "skill_name"])
    counts = df["skill_name"].value_counts()
    df = df[df["skill_name"].isin(counts[counts >= MIN_INTERACTIONS_PER_SKILL].index)]
    df = df.sort_values("order_id").reset_index(drop=True)
    print(f"rows {n0:,} -> {len(df):,}")
    return df[["order_id", "user_id", "skill_name", "correct"]]

data = clean(raw)
print(f"students: {data.user_id.nunique():,} | skills: {data.skill_name.nunique()} | "
      f"overall correct rate: {data.correct.mean():.3f}")

rows 525,534 -> 300,143
students: 4,138 | skills: 99 | overall correct rate: 0.660


### 1c · Quick look

In [4]:
per_skill = (data.groupby("skill_name")
                 .agg(n=("correct", "size"), students=("user_id", "nunique"), p_correct=("correct", "mean"))
                 .sort_values("n", ascending=False))
display(per_skill.head(15)) if "display" in globals() else print(per_skill.head(15))

# Learning curve: correctness by opportunity number (should rise if students learn)
data["opportunity"] = data.groupby(["user_id", "skill_name"]).cumcount() + 1
curve = data[data.opportunity <= 15].groupby("opportunity").correct.mean()
ax = curve.plot(marker="o", title="Mean correctness by practice opportunity")
ax.set_xlabel("opportunity on a skill"); ax.set_ylabel("P(correct)"); plt.show()

                                              n  students  p_correct
skill_name                                                          
Conversion of Fraction Decimals Percents  18740      1225    0.63746
Equation Solving Two or Fewer Steps       17311       961    0.64601
Addition and Subtraction Integers         12741      1226    0.59925
Addition and Subtraction Fractions        11334      1353    0.67699
Percent Of                                 9474      1115    0.59616
Proportion                                 9049       756    0.64151
Ordering Fractions                         8539       882    0.79201
Finding Percents                           7640       771    0.54045
Probability of Two Distinct Events         7453       452    0.49899
Absolute Value                             7340      1002    0.75654
Ordering Positive Decimals                 7317       942    0.74962
Probability of a Single Event              7029       939    0.76298
Multiplication and Division Intege

## 2 · Train/test split **by student**
Splitting by row would leak a student's own history into the test set. Splitting by student mimics the real question: *does a model fit on other students generalise to new ones?* (and later, to Nigerian students).

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED)
tr_idx, te_idx = next(gss.split(data, groups=data["user_id"]))
train, test = data.iloc[tr_idx].copy(), data.iloc[te_idx].copy()
assert set(train.user_id).isdisjoint(test.user_id)
print(f"train rows {len(train):,} ({train.user_id.nunique()} students) | "
      f"test rows {len(test):,} ({test.user_id.nunique()} students)")

train rows 237,992 (3310 students) | test rows 62,151 (828 students)


## 3 · Metrics and baselines

In [6]:
def evaluate(y, p, name):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return dict(model=name, auc=roc_auc_score(y, p),
                rmse=mean_squared_error(y, p) ** 0.5,
                acc=accuracy_score(y, p >= 0.5), logloss=log_loss(y, p))

results = []
y_test = test["correct"].values
results.append(evaluate(y_test, np.full(len(test), train.correct.mean()), "global mean"))
skill_mean = train.groupby("skill_name").correct.mean()
results.append(evaluate(y_test, test.skill_name.map(skill_mean).fillna(train.correct.mean()), "per-skill mean"))
pd.DataFrame(results)

,model,auc,rmse,acc,logloss
0,global mean,0.50000,0.47792,0.64773,0.64936
1,per-skill mean,0.62142,0.46736,0.65391,0.62715


## 4 · BKT from scratch (NumPy)

BKT is a two-state hidden Markov model per skill. The student either **knows** the skill (L) or doesn't. There are four parameters per skill:

| param | meaning |
|---|---|
| `prior` P(L₀) | probability the student already knows it |
| `learn` P(T) | chance of learning it after each opportunity |
| `guess` P(G) | correct answer without knowing it |
| `slip`  P(S) | wrong answer despite knowing it |

For each attempt:

1. **predict** P(correct) = P(L)(1−S) + (1−P(L))G
2. **observe**, Bayes update: if correct, P(L|obs) = P(L)(1−S) / P(correct); if wrong, P(L|obs) = P(L)·S / (1−P(correct))
3. **transition**: P(L) ← P(L|obs) + (1−P(L|obs))·T

This is all the phone has to do offline, which makes BKT a good fit for on-device use.

Fitting: we minimise negative log-likelihood per skill with L-BFGS-B. The code processes all students of a skill at once as a padded matrix. Guess and slip are capped at 0.3 and 0.4 to avoid *degenerate* solutions (e.g. "guess > 1−slip"), which is a known BKT identifiability problem worth a sentence in your report.

In [7]:
BOUNDS = dict(prior=(0.01, 0.99), learn=(0.01, 0.99), guess=(0.01, 0.30), slip=(0.01, 0.40))
NAMES = list(BOUNDS)

def to_matrix(df_skill):
    # one row per student, columns = their attempts in time order (padded)
    seqs = [g.values for _, g in df_skill.groupby("user_id", sort=False)["correct"]]
    T = max(len(s) for s in seqs)
    obs = np.zeros((len(seqs), T), dtype=np.int8); mask = np.zeros((len(seqs), T), bool)
    for i, s in enumerate(seqs):
        obs[i, :len(s)] = s; mask[i, :len(s)] = True
    return obs, mask

def bkt_forward(prior, learn, guess, slip, obs, mask, forget=0.0):
    # returns P(correct) BEFORE each observation (S x T) and final P(known) per student
    p_known = np.full(obs.shape[0], prior)
    preds = np.zeros(obs.shape)
    for t in range(obs.shape[1]):
        p_c = p_known * (1 - slip) + (1 - p_known) * guess
        preds[:, t] = p_c
        o = obs[:, t] == 1
        post = np.where(o, p_known * (1 - slip) / p_c, p_known * slip / (1 - p_c))
        nxt = post * (1 - forget) + (1 - post) * learn
        p_known = np.where(mask[:, t], nxt, p_known)
    return preds, p_known

def nll(theta, obs, mask):
    preds, _ = bkt_forward(*theta, obs, mask)
    p = np.clip(preds[mask], 1e-9, 1 - 1e-9); y = obs[mask]
    return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def fit_skill(df_skill, restarts=3, seed=SEED):
    obs, mask = to_matrix(df_skill)
    r = np.random.default_rng(seed); best = None
    for _ in range(restarts):
        x0 = [r.uniform(*BOUNDS[n]) for n in NAMES]
        res = minimize(nll, x0, args=(obs, mask), method="L-BFGS-B", bounds=[BOUNDS[n] for n in NAMES])
        if best is None or res.fun < best.fun:
            best = res
    return dict(zip(NAMES, map(float, best.x)))

t0 = time.time()
bkt_params = {s: fit_skill(g) for s, g in train.groupby("skill_name")}
# fallback for skills unseen in training (will matter for Nigerian topics with no mapping)
DEFAULT_PARAMS = {n: float(np.median([p[n] for p in bkt_params.values()])) for n in NAMES}
print(f"fitted {len(bkt_params)} skills in {time.time()-t0:.1f}s; default = "
      + ", ".join(f"{k}={v:.3f}" for k, v in DEFAULT_PARAMS.items()))

fitted 99 skills in 48.4s; default = prior=0.678, learn=0.133, guess=0.173, slip=0.206


In [8]:
def predict_bkt(df, params, default=DEFAULT_PARAMS):
    # P(correct) for each row of df, using only that student's earlier answers on the same skill
    out = pd.Series(np.nan, index=df.index)
    for s, g in df.groupby("skill_name"):
        p = params.get(s, default)
        users = list(g.groupby("user_id", sort=False).groups.values())
        obs, mask = to_matrix(g)
        preds, _ = bkt_forward(p["prior"], p["learn"], p["guess"], p["slip"], obs, mask)
        for i, idx in enumerate(users):
            out.loc[idx] = preds[i, :len(idx)]
    return out.values

p_scratch = predict_bkt(test, bkt_params)
results.append(evaluate(y_test, p_scratch, "BKT (NumPy, from scratch)"))
pd.DataFrame(results)

,model,auc,rmse,acc,logloss
0,global mean,0.50000,0.47792,0.64773,0.64936
1,per-skill mean,0.62142,0.46736,0.65391,0.62715
2,"BKT (NumPy, from scratch)",0.72010,0.44096,0.71135,0.57385


## 5 · pyBKT (the library in your plan)
pyBKT fits with EM and supports extras you may want later: `forgets=True`, `multigs=True` (per-problem guess/slip) and `multilearn`. Its default column names already match ASSISTments (`user_id`, `skill_name`, `correct`, `order_id`).

In [9]:
if HAVE_PYBKT:
    t0 = time.time()
    pybkt = PyBKTModel(seed=SEED, num_fits=5)
    pybkt.fit(data=train)
    print(f"pyBKT fit in {time.time()-t0:.1f}s")
    pred_df = pybkt.predict(data=test)
    # pyBKT drops rows for skills it hasn't seen; align on index
    p_pybkt = pred_df["correct_predictions"].reindex(test.index).fillna(train.correct.mean()).values
    results.append(evaluate(y_test, p_pybkt, "BKT (pyBKT)"))
    # example of a variant to compare later:
    # pybkt_f = PyBKTModel(seed=SEED, num_fits=5); pybkt_f.fit(data=train, forgets=True)
    display(pybkt.params().head(12)) if "display" in globals() else print(pybkt.params().head(12))
else:
    print("pyBKT not installed; skipping (pip install pyBKT). The NumPy BKT above is equivalent in model form.")

results_df = pd.DataFrame(results).round(4)
results_df

pyBKT fit in 3000.3s


                                       value
skill                param   class          
Circle Graph         prior   default 0.55673
                     learns  default 0.05758
                     guesses default 0.15404
                     slips   default 0.27992
                     forgets default 0.00000
Percent Of           prior   default 0.65931
                     learns  default 0.11352
                     guesses default 0.07506
                     slips   default 0.28454
                     forgets default 0.00000
Equivalent Fractions prior   default 0.63229
                     learns  default 0.09662


,model,auc,rmse,acc,logloss
0,global mean,0.50000,0.47790,0.64770,0.64940
1,per-skill mean,0.62140,0.46740,0.65390,0.62720
2,"BKT (NumPy, from scratch)",0.72010,0.44100,0.71130,0.57380
3,BKT (pyBKT),0.51170,0.50670,0.58860,0.72350


### Sanity check: does the fit recover the true parameters? (synthetic data only)

In [10]:
if TRUE_PARAMS is not None:
    rec = pd.DataFrame({
        f"{n}_{src}": {s: (TRUE_PARAMS if src == "true" else bkt_params)[s][n] for s in bkt_params}
        for n in NAMES for src in ("true", "fit")})
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    for ax, n in zip(axes, NAMES):
        ax.scatter(rec[f"{n}_true"], rec[f"{n}_fit"]); ax.plot([0, 1], [0, 1], "k--", lw=0.8)
        ax.set_title(n); ax.set_xlabel("true"); ax.set_ylabel("fitted")
    plt.tight_layout(); plt.show()
    print("mean abs error:", {n: round(float((rec[f'{n}_true'] - rec[f'{n}_fit']).abs().mean()), 3) for n in NAMES})

## 6 · Log to MLflow (optional)

In [11]:
if HAVE_MLFLOW:
    mlflow.set_experiment("kt-waec-jamb")
    with mlflow.start_run(run_name=f"bkt-starter-{'synthetic' if USE_SYNTHETIC else 'assist09'}"):
        mlflow.log_params(dict(dataset="synthetic" if USE_SYNTHETIC else "assist09",
                               min_interactions=MIN_INTERACTIONS_PER_SKILL, test_fraction=TEST_FRACTION, seed=SEED))
        for _, row in results_df.iterrows():
            key = row.model.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
            mlflow.log_metric(f"auc__{key}", row.auc); mlflow.log_metric(f"rmse__{key}", row.rmse)
    print("logged to MLflow")

2026/11/23 18:13:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/23 18:13:23 INFO mlflow.store.db.utils: Updating database tables


2026/11/23 18:13:26 INFO mlflow.tracking.fluent: Experiment with name 'kt-waec-jamb' does not exist. Creating a new experiment.


2026/11/23 18:13:26 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 1000'


logged to MLflow


## 7 · Export parameters for the offline PWA
The phone only needs this JSON and the update rule. Here is the JavaScript version, to keep in IndexedDB:

```js
// state: { [skill]: pKnown }, params: loaded from bkt_params.json
function predictCorrect(pK, p) { return pK * (1 - p.slip) + (1 - pK) * p.guess; }
function updateMastery(pK, correct, p) {
  const pc = predictCorrect(pK, p);
  const post = correct ? pK * (1 - p.slip) / pc : pK * p.slip / (1 - pc);
  return post + (1 - post) * p.learn;
}
```
Later, the JSON will map **WAEC/JAMB topics** rather than ASSISTments skill names (that mapping is your notebook 02).

In [12]:
export = {"model": "BKT", "trained_on": "synthetic" if USE_SYNTHETIC else "ASSISTments 2009",
          "default": DEFAULT_PARAMS, "skills": bkt_params}
(OUT_DIR / "bkt_params.json").write_text(json.dumps(export, indent=2))
results_df.to_csv(OUT_DIR / "results.csv", index=False)
print("wrote", OUT_DIR / "bkt_params.json", "and", OUT_DIR / "results.csv")

wrote artifacts/bkt_params.json and artifacts/results.csv


## 8 · Choosing the next question: personalised vs fixed order (demo)
This is a first version of the policy you will compare in the pilot:

- **Fixed order** (baseline): go through topics in curriculum order, giving `N` questions each regardless of answers.
- **BKT path**: stay on the earliest topic in curriculum order whose mastery is below `0.95`, so you move on only once BKT thinks it's learned.

The simulation uses the synthetic "true" students so the outcome can be measured. With real data you'd only have the pilot. Treat this as a check that the logic works, not as evidence.

In [13]:
MASTERY = 0.95

def next_skill_bkt(mastery, curriculum):
    for s in curriculum:
        if mastery[s] < MASTERY:
            return s
    return None  # everything mastered

def simulate(policy, curriculum, true, est, budget=60, n_per_topic=5, seed=0):
    r = np.random.default_rng(seed)
    known = {s: r.random() < true[s]["prior"] for s in curriculum}
    mastery = {s: est[s]["prior"] for s in curriculum}
    for q in range(budget):
        if policy == "fixed":
            s = curriculum[min(q // n_per_topic, len(curriculum) - 1)]
        else:
            s = next_skill_bkt(mastery, curriculum) or curriculum[-1]
        t, p = true[s], est[s]
        correct = r.random() < ((1 - t["slip"]) if known[s] else t["guess"])
        pc = mastery[s] * (1 - p["slip"]) + (1 - mastery[s]) * p["guess"]
        post = mastery[s] * (1 - p["slip"]) / pc if correct else mastery[s] * p["slip"] / (1 - pc)
        mastery[s] = post + (1 - post) * p["learn"]
        if not known[s] and r.random() < t["learn"]:
            known[s] = True
    return np.mean(list(known.values()))  # fraction of topics truly known at the end

if TRUE_PARAMS is not None:
    curriculum = sorted(bkt_params)
    sims = {pol: [simulate(pol, curriculum, TRUE_PARAMS, bkt_params, seed=i) for i in range(500)]
            for pol in ("fixed", "bkt")}
    for pol, v in sims.items():
        print(f"{pol:>5}: mean fraction of topics mastered after 60 questions = {np.mean(v):.3f} (sd {np.std(v):.3f})")

**Note on the result:** on the synthetic data the two policies come out roughly level (about 0.78 vs 0.77 topics mastered). Mastery-based sequencing only pays off when it skips topics a student already knows and spends the saved questions where they're needed. With an even budget of 5 questions per topic and a strict 0.95 threshold, it can instead get stuck on hard topics. Try `MASTERY = 0.85`, a larger budget, or a policy that skips topics already above threshold at the start. This is the kind of design choice to settle **before** the pilot, and it's why the fixed-order comparison is a real test rather than a formality.

## Next steps
- Run with the real ASSISTments CSV (`USE_SYNTHETIC = False`) and record AUCs plus your preprocessing choices.
- Add a `forgets=True` pyBKT variant and compare.
- **Notebook 02:** map your 10–15 WAEC/JAMB topics to ASSISTments skills (keep the mapping table as a CSV; it's a contribution in itself).
- **Notebook 03:** DKT (LSTM, PyTorch) with the same split and metrics so the comparison is fair.
- Collect a small set of Nigerian students' responses early, even 10 to 15 students, so you can compute transfer AUC before the pilot.